# 🏌️ Parameter Golf — Sophonic Quantization (Proof of Concept)**What this does:** Trains the baseline 17M-param GPT model on FineWeb, then runs theSophonic Quantization ablation suite comparing:- Uniform int8 quantization (current competition standard)- int5 base only (measures the damage)- int5 + rank-R low-rank corrections on ALL layers (upper bound)- int5 + corrections on deepest-k layers only (static routing)- int5 + corrections on highest-residual-norm layers (norm routing)- int5 + corrections on random layers (sanity baseline)**Runtime:** ~20 minutes total on a free T4 GPU.**First:** Go to **Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Check GPU

In [ ]:
!nvidia-smiimport torchassert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"print(f"GPU: {torch.cuda.get_device_name(0)}")print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Clone repo & install deps

In [ ]:
# Clone YOUR fork (change the URL to your fork)# If you haven't forked yet, use the upstream directly:!git clone https://github.com/openai/parameter-golf.git%cd parameter-golf# The Colab Python env already has torch, numpy — just need sentencepiece!pip install sentencepiece -q

## 3. Download dataset (1 training shard + full validation)

In [ ]:
# 1 shard = ~100M tokens. Enough to train a meaningful model.# Full validation set always downloads (needed for BPB scoring).# Takes ~2 minutes on Colab's network.!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

## 4. Upload train_sophonic.py

In [ ]:
# Option A: Upload from your machinefrom google.colab import filesuploaded = files.upload()  # Click "Choose Files" and select train_sophonic.py# Option B: If you've pushed to your fork already, skip upload — just:# !cp /content/parameter-golf/train_sophonic.py .# Or pull from your branch:# !git checkout sophonic-quantization

## 5. Run the experimentThis does three things:1. **Trains** the baseline model (~10 min on T4)2. **Evaluates** with standard int8 quantization (the baseline comparison)3. **Runs Sophonic ablation** — int5 base + rank-4 residuals with 5 different routing strategiesThe key output is the comparison table at the very end.

In [ ]:
import os# Configure the runos.environ.update({    "RANK": "0",    "WORLD_SIZE": "1",    "LOCAL_RANK": "0",    "RUN_ID": "sophonic_poc_colab",    "MAX_WALLCLOCK_SECONDS": "300",       # 5 min training (enough for signal on T4)    "VAL_LOSS_EVERY": "0",                # Skip periodic val (saves time), only eval at end    "TRAIN_LOG_EVERY": "50",              # Log every 50 steps    # Sophonic config    "SOPHONIC_ENABLED": "1",    "SOPHONIC_BASE_BITS": "5",            # int5 base    "SOPHONIC_HIGH_BITS": "8",            # int8 corrections    "SOPHONIC_RANK": "4",                 # Rank-4 low-rank residuals    "SOPHONIC_K": "4",                    # Upgrade 4 layers})# Run it!python3 train_sophonic.py

## 6. Understanding the resultsThe Sophonic evaluation prints a table like this:```============================================================SOPHONIC EVALUATION — int5 base + rank-4 residuals, k=4============================================================  Base weights:  X.XX MB  Residuals:     X.XX MB (N matrices)  Mean energy captured by rank-4: XX.X%  Total artifact: XX.XX MB ✅  Method                                     val_bpb     Δ vs int8  ------------------------------------------ --------  --------  Uniform int8 (baseline)                    val_bpb=X.XXXX  Δ=  —  int5 base only                             val_bpb=X.XXXX  Δ=+X.XXXX  int5 + ALL rank-4 corrections              val_bpb=X.XXXX  Δ=+X.XXXX  int5 + static top-4 deepest                val_bpb=X.XXXX  Δ=+X.XXXX  int5 + residual-norm top-4                 val_bpb=X.XXXX  Δ=+X.XXXX  int5 + random 4 layers                     val_bpb=X.XXXX  Δ=+X.XXXX  int5 damage vs int8: +X.XXXX BPB  Rank-4 recovery:     +X.XXXX BPB (XX% of damage)  Net (all corr) vs int8: +X.XXXX BPB```### What to look for:| Signal | Meaning | Next step ||--------|---------|-----------|| `ALL corrections` < int8 | 🎉 **Sophonics works!** | Implement learned router || `ALL corrections` close to int8 (within 0.01) | Promising | Try rank-8, or int6 base || `static` >> `random` | Layer selection matters | Good sign for dynamic routing || `static` ≈ `random` | Selection doesn't matter | Bad sign — all layers equal || `base only` much worse than int8 | int5 is too damaged | Try int6 base instead || Energy captured < 70% | Rank too low | Try rank-8 or rank-16 |

## 7. Quick follow-up experiments

In [ ]:
# ─── Experiment: Higher rank ───os.environ["SOPHONIC_RANK"] = "8"os.environ["RUN_ID"] = "sophonic_rank8"!python3 train_sophonic.py

In [ ]:
# ─── Experiment: int6 base instead of int5 ───os.environ["SOPHONIC_BASE_BITS"] = "6"os.environ["SOPHONIC_RANK"] = "4"os.environ["RUN_ID"] = "sophonic_int6base"!python3 train_sophonic.py

In [ ]:
# ─── Experiment: More layers upgraded ───os.environ["SOPHONIC_BASE_BITS"] = "5"os.environ["SOPHONIC_K"] = "6"os.environ["RUN_ID"] = "sophonic_k6"!python3 train_sophonic.py

## 8. Save resultsDownload your logs for analysis / for your submission write-up.

In [ ]:
# Download all log filesfrom google.colab import filesimport globfor log in glob.glob("logs/*.txt"):    print(f"Downloading {log}")    files.download(log)

---## Notes- **Training only uses 1 shard** (~100M tokens) so absolute BPB numbers will be worse  than the competition (which uses 8B tokens). That's fine — we're comparing *relative*  differences between quantization strategies on the same trained model.- If Colab disconnects mid-run, just re-run from Cell 5. The dataset stays cached.- The full 80-shard dataset would take too long to download on Colab. Use RunPod for that.- If you get OOM on T4, reduce `TRAIN_BATCH_TOKENS` to `262144` (half the default).